<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/02-linear-regression.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 2 — Linear Regression, Properly

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/linear-regression-properly/).

The same delivery world as chapter 1, with a second feature, and the best line found three ways: by calculus, by geometry, and by linear algebra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def mae(predicted, actual):
    return np.mean(np.abs(predicted - actual))

def make_deliveries(n, noise=4.0, seed=0):
    """Distance in km, items in the bag, and the minutes the delivery took."""
    rng = np.random.default_rng(seed)
    km = rng.uniform(0.5, 8.0, n)
    items = rng.integers(1, 7, n)                      # 1 to 6 items
    minutes = 8 + 4.0 * km + 1.5 * items + rng.normal(0, noise, n)
    return km, items, minutes

km, items, minutes = make_deliveries(200, seed=1)
km_test, items_test, minutes_test = make_deliveries(1000, seed=2)
print(f'{len(km)} deliveries; first three:')
for i in range(3):
    print(f'  {km[i]:.1f} km, {items[i]} items -> {minutes[i]:.1f} min')

## 1. One feature, by calculus

In [ ]:
def fit_line(x, y):
    """Slope and intercept that minimise mean squared error, from the two derivatives."""
    x_bar, y_bar = x.mean(), y.mean()
    slope = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
    intercept = y_bar - slope * x_bar
    return slope, intercept

a, b = fit_line(km, minutes)
print(f'minutes = {b:.2f} + {a:.2f} x km')
print(f'the line passes through the means: ({km.mean():.2f}, {minutes.mean():.2f}) -> {b + a * km.mean():.2f}')

In [ ]:
# A bowl: the loss for every slope near the best one, with the intercept held at its best value
slopes = np.linspace(a - 2, a + 2, 9)
for s in slopes:
    loss = np.mean((s * km + b - minutes) ** 2)
    print(f'slope {s:5.2f}  mean squared error {loss:7.2f}')

## 2. Two features, by geometry

In [ ]:
X = np.column_stack([np.ones(len(km)), km, items])     # a column of ones carries the intercept
w, *_ = np.linalg.lstsq(X, minutes, rcond=None)
residuals = minutes - X @ w
print('weights (intercept, km, items):', np.round(w, 3))
print('residuals against each column: ', np.round(X.T @ residuals, 6))
print('mean residual:', round(residuals.mean(), 6))

## 3. Two features, by linear algebra

In [ ]:
XtX = X.T @ X
Xty = X.T @ minutes
w_normal = np.linalg.solve(XtX, Xty)
print('normal equations:', np.round(w_normal, 3))
print('agrees with lstsq:', np.allclose(w_normal, w))
print('true formula    : [8.    4.    1.5  ]')

In [ ]:
def predict(X, w):
    return X @ w

X_test = np.column_stack([np.ones(len(km_test)), km_test, items_test])
print(f'two features on new deliveries: {mae(predict(X_test, w), minutes_test):.2f} min')
print(f'distance alone on new deliveries: {mae(a * km_test + b, minutes_test):.2f} min')
truth = 8 + 4.0 * km_test + 1.5 * items_test
print(f'the true formula                : {mae(truth, minutes_test):.2f} min')

## 4. The one line you would ship

In [ ]:
from sklearn.linear_model import LinearRegression

features = np.column_stack([km, items])
model = LinearRegression().fit(features, minutes)
print(f'intercept {model.intercept_:.3f}, coefficients {np.round(model.coef_, 3)}')
print('same answer as the normal equations:', np.allclose([model.intercept_, *model.coef_], w))

## 5. Reading the coefficients, and when you cannot

In [ ]:
# Two features that say nearly the same thing: distance in km and the same distance in miles, with a little measurement noise
rng = np.random.default_rng(3)
miles = km / 1.609 + rng.normal(0, 0.02, len(km))
X_both = np.column_stack([np.ones(len(km)), km, miles, items])
w_both = np.linalg.solve(X_both.T @ X_both, X_both.T @ minutes)
print('weights (intercept, km, miles, items):', np.round(w_both, 2))
print(f'condition number with both: {np.linalg.cond(X_both):,.0f}   with km only: {np.linalg.cond(X):,.0f}')

X_both_test = np.column_stack([np.ones(len(km_test)), km_test / 1.609, km_test, items_test])[:, [0, 2, 1, 3]]
print(f'predictions still fine: {mae(X_both_test @ w_both, minutes_test):.2f} min on new deliveries')

In [ ]:
# The same fit on a different sample of 200 deliveries
km2, items2, minutes2 = make_deliveries(200, seed=5)
miles2 = km2 / 1.609 + rng.normal(0, 0.02, len(km2))
X2 = np.column_stack([np.ones(len(km2)), km2, miles2, items2])
w2 = np.linalg.solve(X2.T @ X2, X2.T @ minutes2)
print('weights on another sample      :', np.round(w2, 2))
print('km + miles/1.609, first sample :', round(w_both[1] + w_both[2] / 1.609, 2))
print('km + miles/1.609, second sample:', round(w2[1] + w2[2] / 1.609, 2))

## 6. Straight in the parameters, curved in the data

In [ ]:
def bicycle(km):
    return 10 + 3.2 * km + 0.15 * km**2

rng = np.random.default_rng(1)
km_c = rng.uniform(0.5, 8.0, 200)
min_c = bicycle(km_c) + rng.normal(0, 4.0, 200)
km_c_test = rng.uniform(0.5, 8.0, 1000)
min_c_test = bicycle(km_c_test) + rng.normal(0, 4.0, 1000)

X_line = np.column_stack([np.ones(200), km_c])
X_curve = np.column_stack([np.ones(200), km_c, km_c**2])
w_line = np.linalg.lstsq(X_line, min_c, rcond=None)[0]
w_curve = np.linalg.lstsq(X_curve, min_c, rcond=None)[0]
print('line :', np.round(w_line, 3))
print('curve:', np.round(w_curve, 3), '   truth: [10, 3.2, 0.15]')
print(f'line on new orders : {mae(np.column_stack([np.ones(1000), km_c_test]) @ w_line, min_c_test):.2f} min')
print(f'curve on new orders: {mae(np.column_stack([np.ones(1000), km_c_test, km_c_test**2]) @ w_curve, min_c_test):.2f} min')

## 7. Look at the residuals

In [ ]:
res_line = min_c - X_line @ w_line
res_curve = min_c - X_curve @ w_curve
order = np.argsort(km_c)
thirds = np.array_split(order, 3)
print(f'{"":<12} {"short":>7} {"middle":>7} {"long":>7}')
print(f'{"line":<12}', ' '.join(f'{res_line[t].mean():7.2f}' for t in thirds))
print(f'{"curve":<12}', ' '.join(f'{res_curve[t].mean():7.2f}' for t in thirds))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, res, title in zip(axes, [res_line, res_curve], ['straight line', 'line plus km²']):
    ax.scatter(km_c, res, s=10, alpha=0.6)
    ax.axhline(0, color='black', lw=1)
    ax.set_title(f'residuals, {title}')
    ax.set_xlabel('distance (km)')
axes[0].set_ylabel('actual minus predicted (min)')
plt.tight_layout()
plt.show()

## 8. One wild order

In [ ]:
km_o, min_o = km.copy(), minutes.copy()
longest = np.argmax(km_o)
min_o[longest] += 120                                   # the longest trip sat in a broken lift for two hours
a_o, b_o = fit_line(km_o, min_o)
print(f'without the outlier: minutes = {b:.2f} + {a:.2f} x km')
print(f'with the outlier   : minutes = {b_o:.2f} + {a_o:.2f} x km')
print(f'that one order moved the prediction for an 8 km trip by {abs((b_o + a_o * 8) - (b + a * 8)):.1f} min')

## 9. How good is good? R²

In [ ]:
def r_squared(y, y_hat):
    return 1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)

print(f'R² distance only     : {r_squared(minutes_test, a * km_test + b):.3f}')
print(f'R² distance and items: {r_squared(minutes_test, X_test @ w):.3f}')
print(f'R² of the true formula: {r_squared(minutes_test, truth):.3f}   (the ceiling, set by the noise)')

## Try it yourself

1. Change `noise` in `make_deliveries` to 1.0 and to 10.0. How do the fitted weights move, and how does R² move?
2. Add a feature that is pure noise, `rng.normal(0, 1, n)`, to section 3. What weight does it get, and does it change the error on new deliveries?
3. In section 8, replace mean squared error with mean absolute error and find the line by brute force, as in chapter 1. How far does the outlier move that line?